# Strategy KB ablation: Lucario vs Crustle
Evaluation only. Frozen v39 BC weights; the treatment enables exactly `lucario_vs_crustle_wall_breaker`.
Design is seat-matched and policy-seed-matched, not fully simulator-RNG-paired: cabt exposes no simulator seed. Bootstrap uncertainty is limited and this run does not claim causal paired inference.

In [ ]:
RUN_EVAL = True
RUN_TRAINING = False
RUN_SUBMISSION = False
N_PAIRS = 200
RULE_ID = 'lucario_vs_crustle_wall_breaker'
assert RUN_EVAL and not RUN_TRAINING and not RUN_SUBMISSION
assert N_PAIRS == 200
print({'RUN_EVAL': RUN_EVAL, 'RUN_TRAINING': RUN_TRAINING, 'RUN_SUBMISSION': RUN_SUBMISSION, 'pairs': N_PAIRS, 'condition_games': 2 * N_PAIRS, 'rule': RULE_ID})

In [ ]:
import glob, json, os, shutil, subprocess, sys, importlib
from pathlib import Path
if not RUN_EVAL:
    raise RuntimeError('RUN_EVAL must remain enabled for this kernel')
stage_markers = glob.glob('/kaggle/input/**/STAGE_MANIFEST.json', recursive=True)
if len(stage_markers) != 1:
    raise RuntimeError(f'Expected exactly one staged source manifest: {stage_markers}')
stage_root = Path(stage_markers[0]).parent
manifest = json.loads(Path(stage_markers[0]).read_text())
expected_slug = 'hardread-tcg-strategy-kb-ablation-src-v1'
assert manifest['dataset_slug'].endswith(expected_slug)
runtime_root = Path('/kaggle/working/hardread_tcg_strategy_kb_runtime')
if runtime_root.exists(): shutil.rmtree(runtime_root)
shutil.copytree(stage_root, runtime_root)
sys.path.insert(0, str(runtime_root))
# Competition source discovery is intentionally strict: never import a shadow cg package.
cg_games = glob.glob('/kaggle/input/**/cg/game.py', recursive=True)
cg_matches = []
for game_file in cg_games:
    game = Path(game_file).resolve(); cg_dir = game.parent
    if (cg_dir / 'api.py').is_file() and (cg_dir / 'libcg.so').is_file():
        cg_matches.append(cg_dir)
if len(cg_matches) != 1:
    raise RuntimeError(f'Expected one suitable official cg source (game.py + api.py + libcg.so), found: {cg_matches}')
cg_dir = cg_matches[0]; cg_parent = cg_dir.parent
sys.path[:] = [str(cg_parent)] + [p for p in sys.path if p != str(cg_parent)]
os.environ['PYTHONPATH'] = str(cg_parent) + os.pathsep + os.environ.get('PYTHONPATH', '')
for name, module in list(sys.modules.items()):
    if name == 'cg' or name.startswith('cg.'):
        module_path = getattr(module, '__file__', None)
        if module_path is None or not str(Path(module_path).resolve()).startswith(str(cg_dir) + os.sep):
            del sys.modules[name]
cg_game = importlib.import_module('cg.game'); cg_api = importlib.import_module('cg.api'); cg_sim = importlib.import_module('cg.sim')
for module in (sys.modules['cg'], cg_game, cg_api, cg_sim):
    module_path = getattr(module, '__file__', None)
    if module_path is None or not str(Path(module_path).resolve()).startswith(str(cg_dir) + os.sep): raise RuntimeError(f'cg module resolved outside discovered cg_dir: {module_path}')
if not getattr(cg_game, 'lib', None) or not getattr(cg_api, 'lib', None): raise RuntimeError('cg native library did not load')
try:
    native_card_count = len(cg_api.all_card_data())
except Exception as exc:
    raise RuntimeError(f'cg native-load smoke failed before runner invocation: {exc}') from exc
if native_card_count <= 0: raise RuntimeError('cg native-load smoke returned no card data')
weight_matches = glob.glob('/kaggle/input/**/hardread-tcg-bc-v39/weights.npz', recursive=True)
if len(weight_matches) != 1: raise RuntimeError(f'Expected one frozen v39 weights.npz: {weight_matches}')
weights = Path(weight_matches[0])
kb = runtime_root / 'configs/strategy_kb_v1.json'
deck0 = runtime_root / 'configs/deck_lucario.csv'
deck1 = runtime_root / 'configs/deck_crustle.csv'
runner = runtime_root / 'scripts/eval_strategy_kb_ablation.py'
for required in (weights, kb, deck0, deck1, runner): assert required.exists(), required
import src.policy_numpy, src.search_oracle, src.strategy_kb, src.rollout_worker
assert str(runtime_root) in src.rollout_worker.__file__
print('staged source:', runtime_root)
print('weights:', weights)
print('official cg discovery/native smoke complete:', cg_dir, native_card_count)

In [ ]:
preflight = '/kaggle/working/strategy_kb_ablation_preflight.json'
preflight_cmd = [sys.executable, str(runner), '--dry-run', '--games', '2', '--enable-rule', RULE_ID, '--weights', str(weights), '--kb', str(kb), '--deck0-file', str(deck0), '--deck1-file', str(deck1), '--output', preflight]
check = subprocess.run(preflight_cmd, cwd=runtime_root, text=True, capture_output=True)
if check.returncode:
    raise RuntimeError(f'preflight failed ({check.returncode}):\n{check.stdout}\n{check.stderr}')
preflight_result = json.loads(Path(preflight).read_text())
assert preflight_result['status'] == 'validated_manifest_only'
assert preflight_result['rules']['requested_ids'] == [RULE_ID]
assert preflight_result['oracle_config']['baseline_kb_off']['strategy_kb']['type'] == 'disabled'
print('preflight passed:', preflight)

In [ ]:
if not RUN_EVAL: raise RuntimeError('evaluation gate is disabled')
smoke_output = Path('/kaggle/working/strategy_kb_ablation_smoke.json')
smoke_cmd = [sys.executable, str(runner), '--pairs', '2', '--smoke', '--enable-rule', RULE_ID, '--determinizations', '1', '--top-k', '2', '--roll-depth', '2', '--oracle-states', '2', '--weights', str(weights), '--kb', str(kb), '--deck0-file', str(deck0), '--deck1-file', str(deck1), '--output', str(smoke_output)]
smoke = subprocess.run(smoke_cmd, cwd=runtime_root, text=True, capture_output=True, env=os.environ.copy())
if smoke.returncode: raise RuntimeError(f'real simulator smoke failed ({smoke.returncode}):\n{smoke.stdout}\n{smoke.stderr}')
smoke_result = json.loads(smoke_output.read_text())
if smoke_result.get('paired_completed') != 2: raise RuntimeError(f'smoke did not complete 2 pairs: {smoke_result.get("paired_completed")}')
oracle = smoke_result['summaries']['treatment_selected_rules']['oracle']
if oracle['successes'] <= 0: raise RuntimeError(f'smoke had no effective search: {oracle}')
print('smoke rule firing (informational; matchup may not be reached):', oracle['rule_firing'])
print('real simulator smoke artifact:', smoke_output)
output = Path('/kaggle/working/strategy_kb_ablation.json')
run_cmd = [sys.executable, str(runner), '--pairs', str(N_PAIRS), '--enable-rule', RULE_ID, '--weights', str(weights), '--kb', str(kb), '--deck0-file', str(deck0), '--deck1-file', str(deck1), '--output', str(output), '--resume']
completed = subprocess.run(run_cmd, cwd=runtime_root, text=True, capture_output=True)
print(completed.stdout)
if completed.returncode:
    raise RuntimeError(f'evaluation failed ({completed.returncode}):\n{completed.stderr}')
if not output.exists(): raise RuntimeError(f'missing output: {output}')
result = json.loads(output.read_text())
if result.get('status') != 'completed': raise RuntimeError(f'invalid status: {result.get("status")}')
if result.get('paired_completed') != 200: raise RuntimeError(f'expected 200 completed pairs / 400 games: {result.get("paired_completed")}')
if any(value != 'valid' for value in result.get('validity_status', {}).values()): raise RuntimeError(f'invalid final validity: {result.get("validity_status")}')
assert result['schedule'][0]['active_seat'] == 0 and result['schedule'][1]['active_seat'] == 1
assert result['oracle_config']['baseline_kb_off']['strategy_kb']['type'] == 'disabled'
assert result['rules']['treatment_enabled_rule_ids'] == [RULE_ID]
print('artifact:', output)
print(json.dumps({'paired_completed': result['paired_completed'], 'validity_status': result['validity_status'], 'summaries': result['summaries']}, indent=2))